# T-05 — Phase 1: Topic, Dataset, and Baseline

**Project:** Retrieval-Augmented Generation and the Study of Hallucination  
**Scope:** lightweight, reproducible, no training and no fine-tuning

This notebook follows the Phase 1 requirements of *YadYar Lite*. It:

1. defines one narrow research question;
2. prepares a small balanced sample from SQuAD v2;
3. builds one reproducible RAG baseline;
4. runs a tiny sanity check, not the final evaluation;
5. exports everything needed by the separate Phase 2 notebook.

> Run the cells from top to bottom. The last cell downloads `T05_Phase1_Outputs.zip`.


## 1. Narrow project question

> **How accurately can a lightweight RAG baseline answer questions in a controlled SQuAD v2 corpus, and which later failures come from retrieval, generation, or failure to abstain when evidence is absent?**

This is deliberately narrow. We use one public dataset, one embedding retriever, one lightweight generator, and no model training.

### Baseline pipeline

`question → MiniLM embedding → exact FAISS search → top passage → FLAN-T5-base → answer / NO_ANSWER`

The baseline has two inspectable components:

- **Retriever:** finds passages relevant to the question.
- **Generator:** answers only from the best passage and should output `NO_ANSWER` when evidence is missing.


## 2. Dataset and evaluation plan

We use the public **SQuAD v2 validation split**. Every row contains an ID, article title, passage, question, and zero or more gold answers. Questions with no gold answer are intentionally unanswerable, which makes the dataset suitable for studying abstention and unsupported generation.

To keep the experiment small and inspectable, the code creates a deterministic sample of:

- 80 answerable questions;
- 40 unanswerable questions;
- a corpus made from the unique passages associated with those questions.

### Metrics planned for Phase 2

- **Exact Match (EM):** percentage of predictions that match a gold answer after standard SQuAD text normalization.
- **Token F1:** token overlap between a prediction and the best gold answer.

Retrieval hit at top 1/top 3 and abstention rate are saved as diagnostic breakdowns, not treated as replacements for the two primary metrics.

### Planned slices and error categories

- answerable vs. unanswerable questions;
- short vs. long questions;
- gold passage retrieved at rank 1 vs. missed or ranked lower;
- retrieval miss, ignored-evidence candidate, absent-evidence candidate, partial/paraphrase mismatch, and possible multi-answer collapse.

No wrong answer is automatically called a hallucination. Phase 2 produces a review sheet so unsupported claims can be checked manually.


In [ ]:
# Install only the libraries needed by the lightweight baseline.
%pip -q install \
    datasets==5.0.1 \
    sentence-transformers==5.7.0 \
    faiss-cpu==1.15.0 \
    transformers==5.15.0 \
    sentencepiece==0.2.2


In [ ]:
from pathlib import Path
from hashlib import sha1
import json
import random
import shutil

import faiss
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

SEED = 42
N_ANSWERABLE = 80
N_UNANSWERABLE = 40
TOP_K = 3

DATASET_ID = "rajpurkar/squad_v2"
DATASET_SPLIT = "validation"
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
GENERATOR_MODEL = "google/flan-t5-base"

OUTPUT_DIR = Path("/content/T05_Phase1_Outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)


## 3. Load and sample SQuAD v2

The sample is balanced on answerability and fixed by `SEED = 42`, so rerunning the notebook selects the same rows.


In [ ]:
raw_dataset = load_dataset(DATASET_ID, split=DATASET_SPLIT)
full_df = raw_dataset.to_pandas()

def get_answer_texts(answer_object):
    """Return the list of gold answer strings from a SQuAD answers object."""
    if isinstance(answer_object, dict):
        return list(answer_object.get("text", []))
    return []

full_df["gold_answers"] = full_df["answers"].apply(get_answer_texts)
full_df["is_answerable"] = full_df["gold_answers"].apply(bool)

answerable = full_df[full_df["is_answerable"]].sample(
    n=N_ANSWERABLE, random_state=SEED
)
unanswerable = full_df[~full_df["is_answerable"]].sample(
    n=N_UNANSWERABLE, random_state=SEED
)

questions = (
    pd.concat([answerable, unanswerable], ignore_index=True)
    .sample(frac=1, random_state=SEED)
    .reset_index(drop=True)
)

def context_id(text):
    return sha1(text.encode("utf-8")).hexdigest()[:16]

questions["gold_context_id"] = questions["context"].apply(context_id)
questions["gold_answers_json"] = questions["gold_answers"].apply(json.dumps)
questions["question_word_count"] = questions["question"].str.split().str.len()
questions["context_word_count"] = questions["context"].str.split().str.len()

corpus = (
    questions[["gold_context_id", "title", "context"]]
    .drop_duplicates("gold_context_id")
    .rename(columns={"gold_context_id": "context_id"})
    .reset_index(drop=True)
)

question_columns = [
    "id", "title", "question", "gold_answers_json", "is_answerable",
    "gold_context_id", "question_word_count", "context_word_count",
]
questions[question_columns].to_csv(
    OUTPUT_DIR / "phase1_questions.csv", index=False
)
corpus.to_csv(OUTPUT_DIR / "phase1_corpus.csv", index=False)

print(f"Full validation rows: {len(full_df):,}")
print(f"Experiment questions: {len(questions):,}")
print(f"Unique corpus passages: {len(corpus):,}")
display(questions[["question", "is_answerable", "gold_answers"]].head(6))


## 4. Build the retrieval baseline

`all-MiniLM-L6-v2` converts every passage into a ready-made dense vector. Vectors are L2-normalized, so inner product in `IndexFlatIP` acts as cosine similarity. Exact FAISS search is intentionally used because this corpus is small and no index training is needed.


In [ ]:
embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)

corpus_embeddings = embedder.encode(
    corpus["context"].tolist(),
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
).astype("float32")

index = faiss.IndexFlatIP(corpus_embeddings.shape[1])
index.add(corpus_embeddings)

np.save(OUTPUT_DIR / "corpus_embeddings.npy", corpus_embeddings)
faiss.write_index(index, str(OUTPUT_DIR / "corpus.faiss"))

def retrieve(question, k=TOP_K):
    query_vector = embedder.encode(
        [question], normalize_embeddings=True
    ).astype("float32")
    scores, positions = index.search(query_vector, min(k, len(corpus)))
    found = corpus.iloc[positions[0]].copy()
    found["similarity"] = scores[0]
    return found.reset_index(drop=True)

print("FAISS passages indexed:", index.ntotal)


In [ ]:
# Tiny retrieval sanity check: this is not the final Phase 2 evaluation.
sanity_rows = []
for _, row in questions.head(6).iterrows():
    found = retrieve(row["question"])
    retrieved_ids = found["context_id"].tolist()
    sanity_rows.append({
        "id": row["id"],
        "question": row["question"],
        "is_answerable": bool(row["is_answerable"]),
        "gold_context_in_top1": row["gold_context_id"] == retrieved_ids[0],
        "gold_context_in_top3": row["gold_context_id"] in retrieved_ids,
        "top1_title": found.iloc[0]["title"],
        "top1_similarity": float(found.iloc[0]["similarity"]),
    })

retrieval_sanity = pd.DataFrame(sanity_rows)
retrieval_sanity.to_csv(OUTPUT_DIR / "phase1_retrieval_sanity.csv", index=False)
display(retrieval_sanity)


## 5. Configure the generation baseline

FLAN-T5-base is used only for inference. The prompt explicitly restricts the answer to the retrieved passage and defines one abstention token: `NO_ANSWER`.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(GENERATOR_MODEL)
generator = AutoModelForSeq2SeqLM.from_pretrained(GENERATOR_MODEL).to(DEVICE)
generator.eval()

def build_prompt(question, passage):
    return f"""Answer the question using ONLY the passage below.
If the passage does not contain enough evidence, output exactly NO_ANSWER.
Keep the answer short and copy the answer span when possible.

Passage:
{passage}

Question: {question}
Answer:"""

def standardize_answer(text):
    cleaned = text.strip()
    no_answer_forms = {"no_answer", "no answer", "no-answer", "none"}
    return "NO_ANSWER" if cleaned.lower() in no_answer_forms else cleaned

def generate_answer(question, passage):
    prompt = build_prompt(question, passage)
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512,
    ).to(DEVICE)
    with torch.inference_mode():
        output_ids = generator.generate(
            **inputs,
            max_new_tokens=32,
            do_sample=False,
            num_beams=1,
        )
    return standardize_answer(tokenizer.decode(output_ids[0], skip_special_tokens=True))


In [ ]:
# One answerable and one unanswerable example prove that the full baseline runs.
sanity_examples = pd.concat([
    questions[questions["is_answerable"]].head(1),
    questions[~questions["is_answerable"]].head(1),
])

generation_rows = []
for _, row in sanity_examples.iterrows():
    top_passage = retrieve(row["question"], k=1).iloc[0]
    prediction = generate_answer(row["question"], top_passage["context"])
    generation_rows.append({
        "id": row["id"],
        "question": row["question"],
        "gold_answers": json.dumps(row["gold_answers"]),
        "is_answerable": bool(row["is_answerable"]),
        "prediction": prediction,
        "retrieved_title": top_passage["title"],
        "similarity": float(top_passage["similarity"]),
    })

generation_sanity = pd.DataFrame(generation_rows)
generation_sanity.to_csv(OUTPUT_DIR / "phase1_generation_sanity.csv", index=False)
display(generation_sanity)


## 6. Export Phase 1

The export contains the fixed question sample, corpus, embeddings, exact FAISS index, model/config choices, and sanity-check outputs. Phase 2 reads this ZIP instead of resampling the data.


In [ ]:
config = {
    "project_topic": "T-05 Retrieval-Augmented Generation and Hallucination",
    "research_question": (
        "How accurately can a lightweight RAG baseline answer questions in a "
        "controlled SQuAD v2 corpus, and which failures come from retrieval, "
        "generation, or failure to abstain?"
    ),
    "seed": SEED,
    "dataset_id": DATASET_ID,
    "dataset_split": DATASET_SPLIT,
    "full_validation_rows": int(len(full_df)),
    "answerable_questions": N_ANSWERABLE,
    "unanswerable_questions": N_UNANSWERABLE,
    "total_questions": int(len(questions)),
    "unique_passages": int(len(corpus)),
    "embedding_model": EMBEDDING_MODEL,
    "generator_model": GENERATOR_MODEL,
    "retrieval_top_k": TOP_K,
    "generator_passages_used": 1,
    "primary_metrics": ["SQuAD Exact Match", "SQuAD Token F1"],
    "planned_slices": [
        "answerable vs unanswerable",
        "short vs long questions",
        "gold context retrieved at top 1 vs not top 1",
    ],
    "abstention_token": "NO_ANSWER",
}

with open(OUTPUT_DIR / "phase1_config.json", "w", encoding="utf-8") as file:
    json.dump(config, file, indent=2, ensure_ascii=False)

manifest = {
    "files": sorted(path.name for path in OUTPUT_DIR.iterdir()),
    "note": "These are baseline artifacts for the separate Phase 2 notebook.",
}
with open(OUTPUT_DIR / "phase1_manifest.json", "w", encoding="utf-8") as file:
    json.dump(manifest, file, indent=2)

archive = shutil.make_archive(
    "/content/T05_Phase1_Outputs", "zip", root_dir=OUTPUT_DIR
)
print("Created:", archive)

try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Download manually from:", archive)


## References used by the baseline

- [SQuAD v2 dataset card](https://huggingface.co/datasets/rajpurkar/squad_v2)
- [all-MiniLM-L6-v2 model card](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2)
- [FLAN-T5-base model card](https://huggingface.co/google/flan-t5-base)
- [FAISS flat-index documentation](https://github.com/facebookresearch/faiss/wiki/Faiss-indexes)

Any material use of an external AI assistant should also be acknowledged in the final reports, as required by the project document.


## Phase 1 is complete

Keep `T05_Phase1_Outputs.zip` unchanged. The Phase 2 notebook will ask for it.  
The full evaluation, breakdowns, representative errors, limitations, demo, and integration contract belong to Phase 2.
